In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# Basic libraries
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

In [5]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (2000, 8)
Test shape: (500, 7)
Sample submission shape: (500, 2)


In [6]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
test_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
answer_counts = train["answer"].value_counts()

print(answer_counts)

most_count = answer_counts.max()
least_count = answer_counts.min()

print("Answer:", most_count + least_count)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Answer: 814


In [9]:
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text


cleaned_prompts = train["prompt"].apply(clean_text)

all_words = []

for prompt in cleaned_prompts:
    all_words.extend(prompt.split())

vocabulary = set(all_words)

print("Vocabulary size:", len(vocabulary))

Vocabulary size: 859


In [10]:
row1 = train[train["id"] == 1].iloc[0]

cleaned_prompt = clean_text(row1["prompt"])
words = cleaned_prompt.split()


filtered_words = []

for word in words:
    if word not in ENGLISH_STOP_WORDS:
        filtered_words.append(word)

print(filtered_words)
print("Words left:", len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Words left: 13


In [11]:
options = ["A", "B", "C", "D", "E"]

all_text = []

for _, row in train.iterrows():
    all_text.append(row["prompt"])

    for option in options:
        all_text.append(row[option])


tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(all_text)

print("TF-IDF vocabulary size:", len(tfidf.get_feature_names_out()))

TF-IDF vocabulary size: 2762


In [12]:
row1 = train[train["id"] == 1].iloc[0]

prompt_vector = tfidf.transform([row1["prompt"]])
option_vector = tfidf.transform([row1["A"]])

similarity = cosine_similarity(
    prompt_vector,
    option_vector
)[0][0]

print("Similarity:", round(similarity, 4))

Similarity: 0.2328


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

options = ["A", "B", "C", "D", "E"]

all_text = []

for _, row in train.iterrows():
    all_text.append(row["prompt"])

    for option in options:
        all_text.append(row[option])

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(all_text)

print("Vocabulary size:", len(tfidf.get_feature_names_out()))

Vocabulary size: 2762


In [14]:
correct = 0

for _, row in train.iterrows():

    prompt_vector = tfidf.transform([row["prompt"]])

    scores = {}

    for option in options:
        option_vector = tfidf.transform([row[option]])

        score = cosine_similarity(
            prompt_vector,
            option_vector
        )[0][0]

        scores[option] = score

    prediction = max(scores, key=scores.get)

    if prediction == row["answer"]:
        correct += 1


accuracy = (correct / len(train)) * 100

print("Accuracy:", round(accuracy, 2), "%")

Accuracy: 13.7 %


In [15]:
row1 = train[train["id"] == 1].iloc[0]

prompt_vector = tfidf.transform([row1["prompt"]])
option_vector = tfidf.transform([row1["A"]])

similarity = cosine_similarity(
    prompt_vector,
    option_vector
)[0][0]

print("Similarity:", round(similarity, 4))

Similarity: 0.2328


In [16]:
def map_at_3(actual, predictions):

    predictions = predictions[:3]

    if actual not in predictions:
        return 0.0

    rank = predictions.index(actual) + 1

    return 1 / rank

In [17]:
score = map_at_3("C", ["C", "A", "B"])

print(score)

1.0


In [18]:
score = map_at_3("B", ["D", "B", "E"])

print(score)

0.5


In [19]:
answer_order = train["answer"].value_counts().index.tolist()

top3 = answer_order[:3]

print("Top 3:", top3)

scores = []

for answer in train["answer"]:
    score = map_at_3(answer, top3)
    scores.append(score)

majority_map = np.mean(scores)

print("MAP@3:", round(majority_map, 4))

Top 3: ['B', 'C', 'A']
MAP@3: 0.4212


In [20]:
scores = []

for _, row in train.iterrows():

    prompt_vector = tfidf.transform([row["prompt"]])

    similarities = {}

    for option in options:

        option_vector = tfidf.transform([row[option]])

        similarity = cosine_similarity(
            prompt_vector,
            option_vector
        )[0][0]

        similarities[option] = similarity

    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]

    score = map_at_3(row["answer"], top3)

    scores.append(score)


tfidf_map = np.mean(scores)

print("TF-IDF MAP@3:", round(tfidf_map, 4))

TF-IDF MAP@3: 0.3119
